<a href="https://colab.research.google.com/github/indraqubit/-The-Complete-Guide-to-Cross-Platform-JUCE-Audio-Plugin-Development/blob/main/colab/Yue_extend_with_exllama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!nvidia-smi


Sun Oct 25 10:45:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Colab for https://github.com/Mozer/YuE-extend
- It can extend any uploaded mp3
- uses exllama2 module
- flash_attention 2 is not supported for Tesla T4

# Install

In [4]:
!pip install exllamav2
!git clone https://github.com/Mozer/YuE-extend
%cd YuE-extend
!pip install -r requirements.txt
!git clone https://huggingface.co/m-a-p/xcodec_mini_infer
!pip install --upgrade protobuf!pip uninstall -y flash-attn

Cloning into 'YuE-extend'...
remote: Enumerating objects: 853, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 853 (delta 153), reused 132 (delta 132), pack-reused 683 (from 1)
Receiving objects: 100% (853/853), 20.49 MiB | 22.71 MiB/s, done.
Resolving deltas: 100% (442/442), done.
/content/YuE-extend/YuE-extend
  Using cached descript_audiotools-0.7.2-py2.py3-none-any.whl.metadata (3.4 kB)
  Using cached descript_audio_codec-1.0.0-py3-none-any.whl.metadata (7.8 kB)
  Using cached flash_attn-2.8.3.post2.tar.gz (8.5 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'flash-attn' when getting requirements to build wheel

# Download models

In [5]:
# download COT s1 + s2 models
# you can also download ICL model (optional)
!huggingface-cli download Alissonerdx/YuE-s1-7B-anneal-en-cot-exl2-8.0bpw --local-dir workspace/models/YuE-s1-7B-anneal-en-cot-exl2-8.0bpw
!huggingface-cli download Alissonerdx/YuE-s2-1B-general-exl2-8.0bpw --local-dir workspace/models/YuE-s2-1B-general-exl2-8.0bpw


Hint: A new version of huggingface_hub (2.3.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



Don't start UI. Google colab doesn't like UIs. Use command line instead. You can edit lyrics and genre in Files on the left -> Yue-extend/prompt_egs/lyrics.txt (double click). Output mp3 will be saved in workspace/outputs/.

# lyrics2music

In [ ]:
# simple cot music generation. 1 verse (run_n_segments), 30s (max_new_tokens)
!python -u src/yue/infer.py --stage1_use_exl2 --stage1_model "workspace/models/YuE-s1-7B-anneal-en-cot-exl2-8.0bpw" --stage1_cache_size 30000 --stage1_cache_mode FP16 --stage2_use_exl2 --stage2_model "workspace/models/YuE-s2-1B-general-exl2-8.0bpw" --stage2_cache_size 50000 --stage2_cache_mode 'FP16' --genre_txt "prompt_egs/genre.txt" --lyrics_txt "prompt_egs/lyrics.txt" --run_n_segments 1 --output_dir workspace/outputs --cuda_idx 0 --seed 42 --max_new_tokens 3000 --basic_model_config xcodec_mini_infer/final_ckpt/config.yaml --resume_path xcodec_mini_infer/final_ckpt/ckpt_00360000.pth --config_path xcodec_mini_infer/decoders/config.yaml --vocal_decoder_path xcodec_mini_infer/decoders/decoder_131000.pth --inst_decoder_path xcodec_mini_infer/decoders/decoder_151000.pth --resume_after_n -1

Loading exllamav2_ext extension (JIT)...
Building C++/CUDA extension ━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━  26% 0:04:28 -:--:--

# mp3 extend

Trim your mp3 up to 30s. Use "Files" menu on the left to upload your split pop.00001.Vocals.mp3 + pop.00001.Instrumental.mp3 to Yue-extend/prompt_egs. To split use: https://huggingface.co/spaces/theneos/audio-separator or other services.
Don't forget to update genre.txt and put full lyrics into lyrics.txt

In [ ]:
# mp3 extend cot. 1 verse (run_n_segments), 28s (max_new_tokens), 7s from mp3 (extend_mp3_end_time)
!python -u src/yue/infer.py --stage1_use_exl2 --stage1_model "workspace/models/YuE-s1-7B-anneal-en-cot-exl2-8.0bpw" --stage1_cache_size 25000 --stage1_cache_mode FP16 --stage2_use_exl2 --stage2_model "workspace/models/YuE-s2-1B-general-exl2-8.0bpw" --stage2_cache_size 50000 --stage2_cache_mode 'FP16' --genre_txt "prompt_egs/genre.txt" --lyrics_txt "prompt_egs/lyrics.txt" --run_n_segments 1 --output_dir workspace/outputs --cuda_idx 0 --seed 42 --max_new_tokens 2800 --basic_model_config xcodec_mini_infer/final_ckpt/config.yaml --resume_path xcodec_mini_infer/final_ckpt/ckpt_00360000.pth --config_path xcodec_mini_infer/decoders/config.yaml --vocal_decoder_path xcodec_mini_infer/decoders/decoder_131000.pth --inst_decoder_path xcodec_mini_infer/decoders/decoder_151000.pth --extend_mp3 --vocal_track_prompt_path "prompt_egs/pop.00001.Vocals.mp3" --instrumental_track_prompt_path "prompt_egs/pop.00001.Instrumental.mp3" --extend_mp3_end_time 7 --prompt_start_time 0 --prompt_end_time 8 --resume_after_n -1